# Traveling Salesman Problem via Simulated Annealing

This notebook implements a simulated annealing solver for the Traveling Salesman Problem (TSP)
using the *haversine* formula for great-circle distances. It is designed to work with the
`cities*.dat` files from the UVa Computational Physics **salesman** project.

**Features:**

- Read city coordinates from `cities23.dat`, `cities150.dat`, etc.
- Compute total loop distance (including return to start).
- Perform Metropolis updates and simulated annealing.
- Save the optimized path to a `*_opt.dat` file compatible with `routeplot.py`.


In [1]:
import math
import random
import os

R_EARTH = 6371.0  # km

In [2]:
def read_cities(filename):
    """Read cities from a data file.
    
    Expected format per line (after comments):
        longitude latitude "City Name"
        
    Returns
    -------
    list of (lon_deg, lat_deg, name)
    """
    cities = []
    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split(maxsplit=2)
            lon = float(parts[0])
            lat = float(parts[1])
            name = parts[2].strip().strip('"') if len(parts) > 2 else ""
            cities.append((lon, lat, name))
    return cities


def haversine(lon1_deg, lat1_deg, lon2_deg, lat2_deg):
    """Great-circle distance between two points on Earth (in km)."""
    lon1 = math.radians(lon1_deg)
    lat1 = math.radians(lat1_deg)
    lon2 = math.radians(lon2_deg)
    lat2 = math.radians(lat2_deg)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (math.sin(dlat / 2.0) ** 2 +
         math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2.0) ** 2)
    c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
    d = R_EARTH * c
    return d


def tour_length(cities, tour):
    """Total loop length for a tour (including return to start).
    
    Parameters
    ----------
    cities : list of (lon_deg, lat_deg, name)
    tour   : list of indices into `cities`
    """
    total = 0.0
    n = len(tour)
    for i in range(n):
        i1 = tour[i]
        i2 = tour[(i + 1) % n]  # wrap
        lon1, lat1, _ = cities[i1]
        lon2, lat2, _ = cities[i2]
        total += haversine(lon1, lat1, lon2, lat2)
    return total


def propose_swap(tour):
    """Return a new tour obtained by swapping two random positions."""
    n = len(tour)
    i, j = random.sample(range(n), 2)
    new_tour = tour[:]
    new_tour[i], new_tour[j] = new_tour[j], new_tour[i]
    return new_tour


def metropolis_step(cities, tour, current_length, T):
    """One Metropolis step at temperature T.
    
    Returns
    -------
    new_tour, new_length
    """
    candidate = propose_swap(tour)
    new_length = tour_length(cities, candidate)
    dE = new_length - current_length

    if dE < 0:
        return candidate, new_length
    else:
        # Accept with probability exp(-dE/T)
        if random.random() < math.exp(-dE / T):
            return candidate, new_length
        else:
            return tour, current_length


def simulated_annealing(cities, initial_tour, T0=2000.0, alpha=0.995,
                        steps_per_T=2000, Tmin=1e-3):
    """Basic simulated annealing loop for the TSP.
    
    Parameters
    ----------
    cities        : list of (lon, lat, name)
    initial_tour  : list of indices
    T0            : initial temperature
    alpha         : cooling factor (0 < alpha < 1)
    steps_per_T   : Metropolis steps per temperature
    Tmin          : final temperature
    
    Returns
    -------
    best_tour, best_length
    """
    tour = initial_tour[:]
    current_length = tour_length(cities, tour)
    best_tour = tour[:]
    best_length = current_length

    T = T0
    while T > Tmin:
        for _ in range(steps_per_T):
            tour, current_length = metropolis_step(cities, tour, current_length, T)
            if current_length < best_length:
                best_length = current_length
                best_tour = tour[:]
        T *= alpha

    return best_tour, best_length


def write_tour_lonlat(filename, cities, tour):
    """Write an ordered list of city coordinates for routeplot.py.
    
    The file will contain:
        # optimized tour: lon lat
        lon lat
        lon lat
        ...
    """
    with open(filename, "w") as f:
        f.write("# optimized tour: lon lat\n")
        for idx in tour:
            lon, lat, name = cities[idx]
            f.write(f"{lon: .6f} {lat: .6f}\n")

In [3]:
def run_tsp(input_file,
            T0=2000.0, alpha=0.995, steps_per_T=2000, Tmin=1e-3, seed=0):
    """Run simulated annealing TSP on a given city file.
    
    Parameters
    ----------
    input_file : str
        Path to a cities*.dat file.
    Other parameters are SA settings.
    
    Returns
    -------
    initial_length, best_length, optfile
    """
    random.seed(seed)
    cities = read_cities(input_file)
    N = len(cities)
    print(f"Read {N} cities from {input_file}")

    # initial random tour
    initial_tour = list(range(N))
    random.shuffle(initial_tour)

    initial_length = tour_length(cities, initial_tour)
    print(f"Initial tour length: {initial_length:.2f} km") 

    best_tour, best_length = simulated_annealing(
        cities,
        initial_tour,
        T0=T0,
        alpha=alpha,
        steps_per_T=steps_per_T,
        Tmin=Tmin,
    )

    print(f"Optimized tour length: {best_length:.2f} km") 

    base, ext = os.path.splitext(input_file)
    optfile = base + "_opt.dat"
    write_tour_lonlat(optfile, cities, best_tour)
    print(f"Wrote optimized route to {optfile}")
    print(f"INITIAL_LENGTH {initial_length:.6f}")
    print(f"OPTIMIZED_LENGTH {best_length:.6f}")

    return initial_length, best_length, optfile

In [4]:
# Example usage (uncomment and run after placing cities23.dat in this directory):
# init_L, opt_L, optfile = run_tsp("cities23.dat",
#                                  T0=2000.0,
#                                  alpha=0.995,
#                                  steps_per_T=2000,
#                                  Tmin=1e-3,
#                                  seed=1)
# print("Initial:", init_L, "Optimized:", opt_L, "Output file:", optfile)